In [ ]:
import torch
import os

# --- FIX: FORCE INITIALIZATION FOR TORCH._DYNAMO ---
try:
    import torch._dynamo
    import torch._dynamo.decorators
except (ImportError, AttributeError):
    pass
# --------------------------------------------------

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import random
from collections import deque

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps) dengan amplifikasi yang realistis.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 
        
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                            
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            
                            # Scaling Realistis (Max ~12 Mbps untuk menciptakan konflik dengan Bitrate 8Mbps)
                            max_tp = max(throughput_mbps) if throughput_mbps else 1
                            scale_factor = 12.0 / max_tp if max_tp > 0 else 1.0
                            
                            scaled_mbps = [max(0.1, tp * scale_factor) for tp in throughput_mbps]
                            smoothed = pd.Series(scaled_mbps).rolling(window=3, min_periods=1).mean().tolist()
                            
                            self.traces.append({
                                "name": file,
                                "data": smoothed
                            })
                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")
        
        if not self.traces:
            print("⚠️ Folder traces_folder tidak ditemukan atau kosong!")
            self.traces.append({"name": "synth_fallback", "data": [8.0] * 500})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class HybridStreamingEnv(gym.Env):
    """
    Arsitektur Hybrid: RL (Strategic) + Deterministic Controller (Execution).
    """
    def __init__(self, trace_manager):
        super(HybridStreamingEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] # Mbps
        
        # Observation Space: [Buffer, Current_TP, Avg_TP, LastExecAction, BufferTrend, TPTrend]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0, -1.0, -1.0]),
            high=np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0]),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3) # PPO memilih Target Index
        
        # State Internal
        self.state = None 
        self.max_steps = 100
        self.current_step = 0
        self.tp_history = deque(maxlen=3)
        
        # Controller State
        self.current_exec_idx = 1 # Start di MID
        self.upgrade_confidence_counter = 0
        self.exec_history = deque(maxlen=3)
        
        # Parameter Controller
        self.LOW_BUFFER_THRESHOLD = 5.0 # Detik
        self.SAFE_MARGIN = 1.0        # Mbps
        self.UPGRADE_CONFIRM_STEPS = 2

    def _get_normalized_obs(self):
        buffer, current_tp, avg_tp, last_exec, buf_trend, tp_trend = self.state
        
        norm_buffer = np.clip(buffer / 30.0, 0.0, 1.0)           
        norm_current_tp = np.clip(current_tp / 15.0, 0.0, 1.0)               
        norm_avg_tp = np.clip(avg_tp / 15.0, 0.0, 1.0)               
        norm_exec = float(last_exec) / 2.0                   
        norm_buf_trend = np.clip(buf_trend / 10.0, -1.0, 1.0)    
        norm_tp_trend = np.clip(tp_trend / 20.0, -1.0, 1.0)      
        
        return np.array([norm_buffer, norm_current_tp, norm_avg_tp, norm_exec, norm_buf_trend, norm_tp_trend], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options and "trace_idx" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_idx"])
        else:
            trace_name = self.trace_manager.select_random_trace()
            
        initial_tp = self.trace_manager.get_next_bandwidth()
        
        self.tp_history.clear()
        self.exec_history.clear()
        self.upgrade_confidence_counter = 0
        self.current_exec_idx = 1
        
        for _ in range(3):
            self.tp_history.append(initial_tp)
            
        self.state = np.array([10.0, initial_tp, initial_tp, 1.0, 0.0, 0.0], dtype=np.float32)
        self.current_step = 0
        return self._get_normalized_obs(), {"trace": trace_name}

    def step(self, action):
        """
        Input 'action' adalah Target dari RL.
        Fungsi ini menjalankan Deterministic Smoothing Controller untuk menghasilkan Executed Action.
        """
        if isinstance(action, np.ndarray):
            action = action.item()
        
        target_idx = int(action)
        buffer, _, avg_tp, current_idx, _, _ = self.state
        current_idx = int(current_idx)
        
        # Ambil throughput baru
        raw_tp = self.trace_manager.get_next_bandwidth()
        self.tp_history.append(raw_tp)
        new_avg_tp = sum(self.tp_history) / len(self.tp_history)
        
        # Hitung Headroom (Kapasitas sisa terhadap bitrate saat ini)
        headroom = raw_tp - self.bitrates[current_idx]
        
        # --- LAYER 2: DETERMINISTIC SMOOTHING CONTROLLER ---
        executed_idx = current_idx

        # 1. Rate Limiter (Anti Lompat)
        # Batasi perubahan target maksimal 1 level dari posisi saat ini
        if abs(target_idx - current_idx) > 1:
            target_idx = current_idx + np.sign(target_idx - current_idx)

        # 2. Immediate Downgrade Rule (Safety First)
        if headroom < 0 or buffer < self.LOW_BUFFER_THRESHOLD:
            executed_idx = min(target_idx, current_idx - 1)
            executed_idx = max(0, executed_idx) # Jangan di bawah Low
            self.upgrade_confidence_counter = 0
        
        # 3. Upgrade Hysteresis Rule
        elif target_idx > current_idx:
            if headroom > self.SAFE_MARGIN:
                self.upgrade_confidence_counter += 1
            else:
                self.upgrade_confidence_counter = 0
            
            if self.upgrade_confidence_counter >= self.UPGRADE_CONFIRM_STEPS:
                executed_idx = current_idx + 1
                self.upgrade_confidence_counter = 0
            else:
                executed_idx = current_idx # Tahan posisi
                
        # 4. Normal Downgrade / Lateral
        else:
            executed_idx = target_idx
            self.upgrade_confidence_counter = 0

        # 5. Oscillation Guard (Anti Jitter)
        self.exec_history.append(executed_idx)
        if len(self.exec_history) == 3:
            # Deteksi pola gonta-ganti cepat dalam 3 step (misal 1-0-1 atau 0-1-0)
            switches = 0
            for i in range(len(self.exec_history)-1):
                if self.exec_history[i] != self.exec_history[i+1]: switches += 1
            
            if switches >= 2:
                # Ambil nilai tengah (median) untuk memutus osilasi
                executed_idx = int(np.median(list(self.exec_history)))
                # Bekukan history agar tidak loop
                self.exec_history.clear()
                self.exec_history.append(executed_idx)

        # Eksekusi Bitrate
        chosen_bitrate = self.bitrates[executed_idx]
        seg_duration = 5.0
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1))
        stalling = max(0, download_time - buffer)
        
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        buf_trend = new_buffer - buffer
        tp_trend = new_avg_tp - avg_tp

        # --- REWARD STRATEGI (Fokus pada Outcome) ---
        # RL dihargai berdasarkan apa yang BERHASIL dieksekusi oleh sistem hybrid
        reward = float(chosen_bitrate * 1.0)
        
        if stalling > 0:
            reward -= float(stalling * 30.0 + 10.0) # Penalti macet tetap utama
            
        # Switching penalty ringan karena controller sudah menangani smoothing
        if executed_idx != current_idx:
            reward -= 1.0

        # Update State
        self.state = np.array([new_buffer, raw_tp, new_avg_tp, float(executed_idx), buf_trend, tp_trend], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        info = {
            "raw_tp": raw_tp, 
            "buffer": new_buffer,
            "executed_bitrate": chosen_bitrate,
            "target_idx": target_idx,
            "executed_idx": executed_idx
        }
        
        return self._get_normalized_obs(), reward, done, False, info

def run_experiment():
    log_dir = "./rl_logs/"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces:
        print("❌ Tidak ada file trace yang ditemukan.")
        return
        
    # Gunakan HybridStreamingEnv
    env = Monitor(HybridStreamingEnv(tm), log_dir)
    
    print("⏳ Memulai pelatihan agen 'Hybrid-Controller ABR' (300,000 langkah)...")
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.00025,  
                ent_coef=0.05, # Entropi sedikit diturunkan karena Controller sudah stabil
                n_steps=2048,
                batch_size=64)         
    
    model.learn(total_timesteps=500000)
    model.save("hybrid_ndn_ai_model")
    print("✅ Pelatihan selesai. Model hybrid disimpan.")

    print("\n📈 Menghasilkan 8 Grafik Evaluasi (Hybrid Architecture)...")
    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []
        
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            
            history.append({
                'Throughput': info_step['raw_tp'],
                'Buffer': info_step['buffer'], 
                'Exec_Bitrate': info_step['executed_bitrate'],
                'Target_Idx': info_step['target_idx']
            })
            
        df = pd.DataFrame(history)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Grafik Bitrate
        ax1.plot(df.index, df['Throughput'], label='Bandwidth (Mbps)', color='blue', alpha=0.2)
        ax1.step(df.index, df['Exec_Bitrate'], label='Executed Bitrate (Hybrid)', color='red', linewidth=2.5, where='post')
        # Plot target RL sebagai titik tipis untuk melihat perbedaan
        ax1.scatter(df.index, [0.5 if x==0 else 2.5 if x==1 else 8.0 for x in df['Target_Idx']], 
                    label='RL Target', color='purple', s=5, alpha=0.3)
        
        ax1.set_title(f"Evaluasi Hybrid ABR: {trace_name}")
        ax1.set_ylabel("Mbps")
        ax1.legend()
        ax1.grid(alpha=0.2)
        
        # Grafik Buffer
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.15, label='Level Isi Buffer')
        ax2.axhline(y=5, color='orange', linestyle='--', label='Danger Zone (5s)')
        ax2.set_ylabel("Detik")
        ax2.set_xlabel("Segmen")
        ax2.set_ylim(0, 35)
        ax2.legend()
        ax2.grid(alpha=0.2)
        
        plt.tight_layout()
        plot_filename = os.path.join(log_dir, f"hybrid_result_{trace_name}.png")
        plt.savefig(plot_filename)
        plt.close()
        print(f"  > Gambar Hybrid tersimpan: {plot_filename}")

    try:
        x, y = ts2xy(load_results(log_dir), 'timesteps')
        plt.figure(figsize=(10, 5))
        plt.plot(x, pd.Series(y).rolling(window=100).mean(), color='crimson')
        plt.title("Track Record Pelatihan (Hybrid Architecture)")
        plt.xlabel("Total Langkah")
        plt.ylabel("Reward")
        plt.savefig(os.path.join(log_dir, "training_progress_hybrid.png"))
        plt.close()
    except:
        pass

if __name__ == "__main__":
    run_experiment()

⏳ Memulai pelatihan agen 'Hybrid-Controller ABR' (300,000 langkah)...
Using cpu device
Wrapping the env in a DummyVecEnv.


AttributeError: 'HybridStreamingEnv' object has no attribute 'exec_history'